In [1]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage , HumanMessage
from langgraph.checkpoint.memory import MemorySaver

In [2]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [3]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage] , add_messages]

In [4]:
graph = StateGraph(ChatState)

In [5]:
def chat_node(state: ChatState):
    messages = state['messages']
    
    response = llm.invoke(messages)
    
    return {'messages' : [response]}    

In [6]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)
graph.add_node('chat_node' , chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node' , END)

chatbot = graph.compile(checkpointer = checkpointer)

In [7]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [11]:
thread_id = '1'
initial_state = {
    'messages' : [HumanMessage(content='which llm model you are using')]
}
chatbot.invoke(initial_state)['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [10]:
while True:
    user_message = input('Type here: ')
    print('User Message:', user_message)
    
    # 1. Added () to call .lower() as a method
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
    
    config = {'configurable' : {'thread_id' : thread_id}}
    
    # 2. Pass a list of messages directly into invoke() instead of a dict
    response = chatbot.invoke(
        {'messages':[HumanMessage(content=user_message)]},config=config)
    
    # 3. BaseChatModel return objects directly expose .content
    print('AI Message:', response['messages'][-1].content)

User Message: jai shree hari
AI Message: Jai Shree Hari! How may I assist you today?
User Message: how are you
AI Message: I'm just a program, but I'm here and ready to help you! How can I assist you today?
User Message: exit


In [12]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Jai mata di', additional_kwargs={}, response_metadata={}, id='c826af69-383b-44e8-a0ac-b4a296b72684'), AIMessage(content='Jai Mata Di! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 11, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ce1b7ea74c', 'id': 'chatcmpl-EPCDrzzSRYhJfnwHSUXMX2ZpmkZQo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0b0e0-0efb-7a00-a844-01e134e1b4bb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 12, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cach